In [1]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from pyspark.sql.functions import col, sum, count, max
from datetime import datetime, timedelta

from builtins import round 
spark = SparkSession.builder.getOrCreate()


# ---------- RUN TYPE ----------
if 'run_type' not in globals():
    run_type = "manual"

# ---------- RUN ID ----------
if 'run_id' not in globals():
    run_id = datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")

print("RUN TYPE:", run_type)
print("RUN ID:", run_id)

StatementMeta(, ad7a205a-e72c-4e03-b92e-52056cd52127, 3, Finished, Available, Finished, False)

RUN TYPE: manual
RUN ID: 2026-05-04 05:38:16


In [2]:
df1 = spark.read.table("Olist_Silver_Lakehouse.silver_olist_customers")
df2 = spark.read.table("Olist_Silver_Lakehouse.silver_olist_geolocation")
df3 = spark.read.table("Olist_Silver_Lakehouse.silver_olist_order_items")
df4 = spark.read.table("Olist_Silver_Lakehouse.silver_olist_order_payments")
df5 = spark.read.table("Olist_Silver_Lakehouse.silver_olist_order_reviews")
df6 = spark.read.table("Olist_Silver_Lakehouse.silver_olist_orders")
df7 = spark.read.table("Olist_Silver_Lakehouse.silver_olist_products")
df8 = spark.read.table("Olist_Silver_Lakehouse.silver_olist_sellers")
df9 = spark.read.table("Olist_Silver_Lakehouse.silver_olist_product_category_translation")


df10 = spark.read.table("Olist_Silver_Lakehouse.ref_country_state")
df11 = spark.read.table("Olist_Silver_Lakehouse.ref_city_master")

StatementMeta(, ad7a205a-e72c-4e03-b92e-52056cd52127, 4, Finished, Available, Finished, False)

In [3]:
def normalize_column(column):
    column = regexp_replace(column, "á|à|ã|â|ä", "a")
    column = regexp_replace(column, "é|è|ê|ë", "e")
    column = regexp_replace(column, "í|ì|î|ï", "i")
    column = regexp_replace(column, "ó|ò|õ|ô|ö", "o")
    column = regexp_replace(column, "ú|ù|û|ü", "u")
    column = regexp_replace(column, "ç", "c")

    column = lower(trim(column))
    column = regexp_replace(column, "[^a-z0-9 ]", "")
    column = regexp_replace(column, " +", " ")
    return column

df1_clean = (
    df1
    .withColumn("customer_state_clean", trim(upper(col("customer_state"))))
    .withColumn("customer_city_clean", normalize_column(col("customer_city")))
)
df10_clean = (
    df10
    .withColumn("state_code", trim(upper(col("state_code"))))
    .withColumn("state_name_clean", normalize_column(col("state_name")))
)
df11_clean = (
    df11
    .withColumn("state_code", trim(upper(col("state_code"))))
    .withColumn("city_name_clean", normalize_column(col("city_name")))
)

StatementMeta(, ad7a205a-e72c-4e03-b92e-52056cd52127, 5, Finished, Available, Finished, False)

#### **Table : order_items**

In [4]:
from pyspark.sql import Row
from pyspark.sql.functions import col, count as spark_count, sum as spark_sum, current_timestamp
from datetime import datetime
from builtins import round as py_round
print("RUN ID:", run_id)
print("RUN TYPE:", run_type)
dq_log = []

# -------------------------------
# Total count
# -------------------------------
total_count = df3.count()

if total_count == 0:
    raise Exception("Order items table is empty")

# -------------------------------
# Helper function
# -------------------------------
def log_check(check_name, issue_count, issue_pct, threshold_error, threshold_warning):
    if threshold_error is not None and issue_pct > threshold_error:
        status = "ERROR"
    elif issue_pct > threshold_warning:
        status = "WARNING"
    else:
        status = "PASS"

    dq_log.append(Row(
        table_name="order_items",
        check_name=check_name,
        status=status,
        percentage=py_round(float(issue_pct), 2),
        record_count=int(issue_count)
    ))

# -------------------------------
# 1. Critical Null Checks
# -------------------------------
null_id = df3.filter(
    col("order_id").isNull() |
    col("order_item_id").isNull() |
    col("product_id").isNull() |
    col("seller_id").isNull()
).count()

null_id_pct = (null_id / total_count) * 100
log_check("null_check", null_id, null_id_pct, 1, 0)

# -------------------------------
# 2. Price Validation
# -------------------------------
invalid_price = df3.filter(col("is_price_valid") == 0).count()
price_pct = (invalid_price / total_count) * 100
log_check("invalid_price", invalid_price, price_pct, 3, 0)

# -------------------------------
# 3. Freight Validation
# -------------------------------
invalid_freight = df3.filter(col("is_freight_value_valid") == 0).count()
freight_pct = (invalid_freight / total_count) * 100
log_check("invalid_freight", invalid_freight, freight_pct, 3, 0)

# -------------------------------
# 4. Delivered but Missing Price
# -------------------------------
delivered_price_missing = df3.filter(
    col("is_delivered_price_missing") == 1
).count()
delivered_price_pct = (delivered_price_missing / total_count) * 100
log_check("delivered_price_missing", delivered_price_missing, delivered_price_pct, 1, 0)

# -------------------------------
# 5. Duplicate Check
# Grain = (order_id, order_item_id)
# -------------------------------
dup_count = df3.groupBy("order_id", "order_item_id") \
    .agg(spark_count("*").alias("cnt")) \
    .filter(col("cnt") > 1).count()

dup_pct = (dup_count / total_count) * 100
log_check("duplicate_check", dup_count, dup_pct, 1, 0)

# -------------------------------
# 6. Payment Validation
# Only for completed/delivered orders
# -------------------------------
df_payment_agg = df4.groupBy("order_id") \
    .agg(spark_sum("payment_value").alias("total_payment"))

df_orders_payment = df6.select("order_id", "order_status")

df_check = df3.select("order_id").distinct() \
    .join(df_orders_payment, "order_id", "left") \
    .join(df_payment_agg, "order_id", "left")

invalid_payment = df_check.filter(
    (col("order_status").isin("delivered", "shipped")) &
    col("total_payment").isNull()
).count()

payment_pct = (invalid_payment / df_check.count()) * 100
log_check("missing_payment_for_completed_orders", invalid_payment, payment_pct, 1, 0)

# -------------------------------
# Write to audit table
# -------------------------------
dq_df = spark.createDataFrame(dq_log) \
    .withColumn("run_id", lit(run_id)) \
    .withColumn("run_timestamp", current_timestamp()) \
    .withColumn("run_type", lit(run_type)) \
    .withColumn("run_date", to_date(current_timestamp()))

dq_df.write.mode("append").saveAsTable("Olist_Gold_Lakehouse.gold_data_quality_log")

# -------------------------------
# Print warnings
# -------------------------------
warnings = [r for r in dq_log if r.status == "WARNING"]
for w in warnings:
    print(f"⚠️  {w.check_name}: {w.percentage:.2f}% ({w.record_count} records)")

# -------------------------------
# Final validation
# -------------------------------
error_count = dq_df.filter(col("status") == "ERROR").count()

if error_count > 0:
    error_checks = [r.check_name for r in dq_log if r.status == "ERROR"]
    raise Exception(f"Data Quality Failed for order_items table: {' | '.join(error_checks)}")
else:
    print("✅ Order Items Data Quality Passed")

StatementMeta(, ad7a205a-e72c-4e03-b92e-52056cd52127, 6, Finished, Available, Finished, False)

RUN ID: 2026-05-04 05:38:16
RUN TYPE: manual
⚠️  invalid_price: 1.92% (2159 records)
⚠️  delivered_price_missing: 0.98% (1107 records)
⚠️  missing_payment_for_completed_orders: 0.00% (3 records)
✅ Order Items Data Quality Passed


#### **Table : orders**

In [5]:
from pyspark.sql import Row
from pyspark.sql.functions import col, count as spark_count, sum as spark_sum, current_timestamp
from datetime import datetime
from builtins import round as py_round

print("RUN ID:", run_id)
print("RUN TYPE:", run_type)
dq_log = []

# -------------------------------
# Total count
# -------------------------------
total_count = df6.count()

if total_count == 0:
    raise Exception("Orders table is empty")

# -------------------------------
# Helper function
# -------------------------------
def log_check(check_name, issue_count, issue_pct, threshold_error, threshold_warning):
    if threshold_error is not None and issue_pct > threshold_error:
        status = "ERROR"
    elif issue_pct > threshold_warning:
        status = "WARNING"
    else:
        status = "PASS"

    dq_log.append(Row(
        table_name="orders",
        check_name=check_name,
        status=status,
        percentage=py_round(float(issue_pct), 2),
        record_count=int(issue_count)
    ))

# -------------------------------
# 1. Null Checks
# -------------------------------
null_order_id = df6.filter(col("order_id").isNull()).count()
null_order_id_pct = (null_order_id / total_count) * 100
log_check("null_order_id", null_order_id, null_order_id_pct, 1, 0)

null_customer_id = df6.filter(col("customer_id").isNull()).count()
null_customer_id_pct = (null_customer_id / total_count) * 100
log_check("null_customer_id", null_customer_id, null_customer_id_pct, 1, 0)

# -------------------------------
# 2. Duplicate Check
# -------------------------------
duplicate_count = df6.groupBy("order_id") \
    .agg(spark_count("*").alias("cnt")) \
    .filter(col("cnt") > 1) \
    .count()
duplicate_pct = (duplicate_count / total_count) * 100
log_check("duplicate_order_id", duplicate_count, duplicate_pct, 1, 0)

# -------------------------------
# 3. Purchase after approval
# -------------------------------
invalid_purchase = df6.filter(col("is_purchase_after_approved") == 1).count()
purchase_pct = (invalid_purchase / total_count) * 100
log_check("purchase_after_approval", invalid_purchase, purchase_pct, 5, 0)

# -------------------------------
# 4. Approval after carrier
# -------------------------------
invalid_approval = df6.filter(col("is_approval_after_carrier") == 1).count()
approval_pct = (invalid_approval / total_count) * 100
log_check("approval_after_carrier", invalid_approval, approval_pct, 5, 0)

# -------------------------------
# 5. Carrier after customer
# -------------------------------
invalid_shipping = df6.filter(col("is_carrier_after_customer") == 1).count()
shipping_pct = (invalid_shipping / total_count) * 100
log_check("carrier_after_customer", invalid_shipping, shipping_pct, 5, 0)

# -------------------------------
# 6. Missing payment for completed orders
# -------------------------------
df_payment_agg = df4.groupBy("order_id") \
    .agg(spark_count("*").alias("payment_count"))

df_check = df6.select("order_id", "order_status") \
    .join(df_payment_agg, "order_id", "left")

missing_payment = df_check.filter(
    (col("order_status") == "delivered") &
    col("payment_count").isNull()
).count()

missing_payment_base = df_check.filter(
    col("order_status") == "delivered"
).count()

missing_payment_pct = (missing_payment / missing_payment_base) * 100
log_check("missing_payment_for_completed_orders", missing_payment, missing_payment_pct, 1, 0)

# -------------------------------
# Write to audit table
# -------------------------------
dq_df = spark.createDataFrame(dq_log) \
    .withColumn("run_id", lit(run_id)) \
    .withColumn("run_timestamp", current_timestamp()) \
    .withColumn("run_type", lit(run_type)) \
    .withColumn("run_date", to_date(current_timestamp()))

dq_df.write.mode("append").saveAsTable("Olist_Gold_Lakehouse.gold_data_quality_log")

# -------------------------------
# Print warnings
# -------------------------------
warnings = [r for r in dq_log if r.status == "WARNING"]
for w in warnings:
    print(f"⚠️  {w.check_name}: {w.percentage:.2f}% ({w.record_count} records)")

# -------------------------------
# Final validation
# -------------------------------
error_count = dq_df.filter(col("status") == "ERROR").count()

if error_count > 0:
    error_checks = [r.check_name for r in dq_log if r.status == "ERROR"]
    raise Exception(f"Data Quality Failed for orders table: {' | '.join(error_checks)}")
else:
    print("✅ Orders Data Quality Passed")

StatementMeta(, ad7a205a-e72c-4e03-b92e-52056cd52127, 7, Finished, Available, Finished, False)

RUN ID: 2026-05-04 05:38:16
RUN TYPE: manual
⚠️  null_customer_id: 1.00% (992 records)
⚠️  purchase_after_approval: 0.99% (987 records)
⚠️  approval_after_carrier: 1.37% (1359 records)
⚠️  carrier_after_customer: 0.94% (936 records)
⚠️  missing_payment_for_completed_orders: 0.00% (3 records)
✅ Orders Data Quality Passed


#### **Table : order_payments**

In [6]:
import builtins
from pyspark.sql import Row
from pyspark.sql.functions import (
    col,
    sum as spark_sum,
    abs as spark_abs,  
    max as spark_max,
    count as spark_count,
    current_timestamp
)
from datetime import datetime
from builtins import round as py_round

print("RUN ID:", run_id)
dq_log = []

# -------------------------------
# Total count
# -------------------------------
total_count = df4.count()

if total_count == 0:
    raise Exception("Payments table is empty")

# -------------------------------
# Helper function
# -------------------------------
def log_check(check_name, issue_count, issue_pct, threshold_error, threshold_warning):
    if threshold_error is not None and issue_pct > threshold_error:
        status = "ERROR"
    elif issue_pct > threshold_warning:
        status = "WARNING"
    else:
        status = "PASS"
    dq_log.append(Row(
        table_name="payments",
        check_name=check_name,
        status=status,
        percentage=py_round(float(issue_pct), 2),
        record_count=int(issue_count)
    ))

# -------------------------------
# 1. Null Checks
# -------------------------------
null_count = df4.filter(
    col("order_id").isNull() |
    col("payment_sequential").isNull() |
    col("payment_type").isNull() |
    col("payment_installments").isNull() |
    col("payment_value").isNull()
).count()
null_pct = (null_count / total_count) * 100
log_check("null_check", null_count, null_pct, 1, 0)

# -------------------------------
# 2. Duplicate Check
# -------------------------------
duplicate_count = df4.groupBy("order_id", "payment_sequential") \
    .agg(spark_count("*").alias("cnt")) \
    .filter(col("cnt") > 1).count()
duplicate_pct = (duplicate_count / total_count) * 100
log_check("duplicate_check", duplicate_count, duplicate_pct, 1, 0)

# -------------------------------
# 3. Sequential Consistency
# -------------------------------
total_orders = df4.select("order_id").distinct().count()

seq_df_count = df4.groupBy("order_id") \
    .agg(spark_count("*").alias("records"))

seq_df = df4.groupBy("order_id") \
    .agg(spark_max(col("payment_sequential").cast("int")).alias("max_seq")) \
    .join(seq_df_count, "order_id") \
    .filter(col("max_seq") != col("records"))

seq_issue = seq_df.count()
seq_pct = (seq_issue / total_orders) * 100 if total_orders > 0 else 0.0
log_check("payment_sequence_consistency", seq_issue, seq_pct, None, 0)

# -------------------------------
# 4. Referential Integrity
# -------------------------------
ref_issue = df4.join(df6, "order_id", "left_anti").count()
ref_pct = (ref_issue / total_count) * 100
log_check("missing_orders", ref_issue, ref_pct, 1, 0)

# -------------------------------
# 5. Negative Values
# -------------------------------
neg_count = df4.filter(col("payment_value") < 0).count()
neg_pct = (neg_count / total_count) * 100
log_check("negative_payment_values", neg_count, neg_pct, 0, 0)

# -------------------------------
# 6. Revenue Validation
# -------------------------------
from pyspark.sql.functions import col, sum as spark_sum, abs as spark_abs

# Delivered orders
valid_orders = df6.filter(col("order_status") == "delivered").select("order_id")

# Order total
order_total = df3.join(valid_orders, "order_id") \
    .filter(
        (col("is_price_valid") == 1) &
        (col("is_freight_value_valid") == 1)
    ) \
    .agg(
    spark_sum(
        col("price").cast("double") + col("freight_value").cast("double")
    ).alias("total_revenue")
).first()[0] or 0.0

# Payment total
payment_total = df4.join(valid_orders, "order_id").agg(
    spark_sum("payment_value")
).first()[0] or 0.0

payment_total = float(payment_total or 0.0)
order_total = float(order_total or 0.0)

# Difference
diff = builtins.abs(payment_total - order_total)
diff_pct = (diff / order_total * 100) if order_total != 0 else 0.0

log_check("revenue_validation", diff, diff_pct, None, 0)

# -------------------------------
# Write to audit table
# -------------------------------
dq_df = spark.createDataFrame(dq_log) \
    .withColumn("run_id", lit(run_id)) \
    .withColumn("run_timestamp", current_timestamp()) \
    .withColumn("run_type", lit(run_type)) \
    .withColumn("run_date", to_date(current_timestamp()))

dq_df.write.mode("append").saveAsTable("Olist_Gold_Lakehouse.gold_data_quality_log")

# -------------------------------
# Print warnings
# -------------------------------
for r in dq_log:
    if r.status == "WARNING":
        print(f"⚠️  {r.check_name}: {r.percentage:.2f}% ({r.record_count} records)")

# -------------------------------
# Final validation
# -------------------------------
error_count = dq_df.filter(col("status") == "ERROR").count()
if error_count > 0:
    error_checks = [r.check_name for r in dq_log if r.status == "ERROR"]
    raise Exception(f"Data Quality Failed for payments table: {' | '.join(error_checks)}")
else:
    print("✅ Payments Data Quality Passed")

StatementMeta(, ad7a205a-e72c-4e03-b92e-52056cd52127, 8, Finished, Available, Finished, False)

RUN ID: 2026-05-04 05:38:16
⚠️  payment_sequence_consistency: 0.08% (78 records)
⚠️  revenue_validation: 1.92% (290749 records)
✅ Payments Data Quality Passed


#### **Table : Order_reviews**

In [7]:
from pyspark.sql import Row
from pyspark.sql.functions import col, current_timestamp
from datetime import datetime
from builtins import round as py_round
print("RUN ID:", run_id)
dq_log = []

total_count = df5.count()
if total_count == 0:
    raise Exception("Review table is empty")

def log_check(check_name, issue_count, issue_pct, threshold_error, threshold_warning):
    if threshold_error is not None and issue_pct > threshold_error:
        status = "ERROR"
    elif issue_pct > threshold_warning:
        status = "WARNING"
    else:
        status = "PASS"
    dq_log.append(Row(
        table_name="reviews",
        check_name=check_name,
        status=status,
        percentage=py_round(float(issue_pct), 2),
        record_count=int(issue_count)
    ))

# 1. Critical Null Checks
null_critical = df5.filter(
    col("review_id").isNull() |
    col("order_id").isNull() |
    col("review_score").isNull()
).count()
null_critical_pct = (null_critical / total_count) * 100
log_check("null_critical_fields", null_critical, null_critical_pct, 1, 0)

# 2. Optional Null Checks (dates)
null_dates = df5.filter(
    col("review_creation_date").isNull() |
    col("review_answer_timestamp").isNull()
).count()
null_dates_pct = (null_dates / total_count) * 100
log_check("null_review_dates", null_dates, null_dates_pct, None, 0)

# 3. Date Validation
invalid_date_flag = df5.filter(col("is_answer_after_review") == 0).count()
invalid_date_pct = (invalid_date_flag / total_count) * 100
log_check("invalid_review_date_sequence", invalid_date_flag, invalid_date_pct, 1, 0)

# 4. Review Score Validation
invalid_score_flag = df5.filter(col("is_review_score_valid") == 0).count()
invalid_score_pct = (invalid_score_flag / total_count) * 100
log_check("invalid_review_score", invalid_score_flag, invalid_score_pct, 1, 0)

# Write to audit table
dq_df = spark.createDataFrame(dq_log) \
    .withColumn("run_id", lit(run_id)) \
    .withColumn("run_timestamp", current_timestamp()) \
    .withColumn("run_type", lit(run_type)) \
    .withColumn("run_date", to_date(current_timestamp()))

dq_df.write.mode("append").saveAsTable("Olist_Gold_Lakehouse.gold_data_quality_log")

# Print warnings
for r in dq_log:
    if r.status == "WARNING":
        print(f"⚠️  {r.check_name}: {r.percentage:.2f}% ({r.record_count} records)")

# Final validation
error_count = dq_df.filter(col("status") == "ERROR").count()
if error_count > 0:
    error_checks = [r.check_name for r in dq_log if r.status == "ERROR"]
    raise Exception(f"Data Quality Failed for reviews table: {' | '.join(error_checks)}")
else:
    print("✅ Reviews Data Quality Passed")

StatementMeta(, ad7a205a-e72c-4e03-b92e-52056cd52127, 9, Finished, Available, Finished, False)

RUN ID: 2026-05-04 05:38:16
✅ Reviews Data Quality Passed


#### **Table : Customers**

In [8]:
print("RUN ID:", run_id)
dq_log = []

total_count = df1.count()
if total_count == 0:
    raise Exception("Customer table is empty")

def log_check(check_name, issue_count, issue_pct, threshold_error, threshold_warning):
    if threshold_error is not None and issue_pct > threshold_error:
        status = "ERROR"
    elif issue_pct > threshold_warning:
        status = "WARNING"
    else:
        status = "PASS"
    dq_log.append(Row(
        table_name="customers",
        check_name=check_name,
        status=status,
        percentage=py_round(float(issue_pct), 2),
        record_count=int(issue_count)
    ))

# 1. Null Customer ID
null_id = df1.filter(col("customer_id").isNull()).count()
null_id_pct = (null_id / total_count) * 100
log_check("null_customer_id", null_id, null_id_pct, 1, 0)

# 2. Null Attributes
null_attr = df1.filter(
    col("customer_zip_code_prefix").isNull() |
    col("customer_city").isNull() |
    col("customer_state").isNull()
).count()
null_attr_pct = (null_attr / total_count) * 100
log_check("null_customer_attributes", null_attr, null_attr_pct, None, 5)

# 3. Duplicate Customer ID
dup_id = df1.groupBy("customer_id") \
    .agg(spark_count("*").alias("cnt")) \
    .filter(col("cnt") > 1).count()
dup_id_pct = (dup_id / total_count) * 100
log_check("duplicate_customer_id", dup_id, dup_id_pct, 1, 0)

# 4. Duplicate Customer Unique ID
dup_unique = df1.groupBy("customer_unique_id") \
    .agg(spark_count("*").alias("cnt")) \
    .filter(col("cnt") > 1).count()
dup_unique_pct = (dup_unique / total_count) * 100
log_check("duplicate_customer_unique_id", dup_unique, dup_unique_pct, None, 5)

# 5. Location Validation
invalid_location = df1.filter(
    (col("is_state_valid") == 0) |
    (col("is_city_valid") == 0) |
    (col("is_state_city_valid") == 0)
).count()
invalid_location_pct = (invalid_location / total_count) * 100
log_check("invalid_customer_location", invalid_location, invalid_location_pct, None, 5)

# -------------------------------
# Write to audit table
# -------------------------------
dq_df = spark.createDataFrame(dq_log) \
    .withColumn("run_id", lit(run_id)) \
    .withColumn("run_timestamp", current_timestamp()) \
    .withColumn("run_type", lit(run_type)) \
    .withColumn("run_date", to_date(current_timestamp()))

dq_df.write.mode("append").saveAsTable("Olist_Gold_Lakehouse.gold_data_quality_log")

# Print warnings
for r in dq_log:
    if r.status == "WARNING":
        print(f"⚠️  {r.check_name}: {r.percentage:.2f}% ({r.record_count} records)")

# Final validation
error_count = dq_df.filter(col("status") == "ERROR").count()
if error_count > 0:
    error_checks = [r.check_name for r in dq_log if r.status == "ERROR"]
    raise Exception(f"Data Quality Failed for customers table: {' | '.join(error_checks)}")
else:
    print("✅ Customers Data Quality Passed")

StatementMeta(, ad7a205a-e72c-4e03-b92e-52056cd52127, 10, Finished, Available, Finished, False)

RUN ID: 2026-05-04 05:38:16
✅ Customers Data Quality Passed


#### **Table : Products**

In [9]:
from pyspark.sql import Row
from pyspark.sql.functions import col, current_timestamp
from datetime import datetime
from builtins import round as py_round
print("RUN ID:", run_id)
dq_log = []

total_count = df7.count()
if total_count == 0:
    raise Exception("Products table is empty")

def log_check(check_name, issue_count, issue_pct, threshold_error, threshold_warning):
    if threshold_error is not None and issue_pct > threshold_error:
        status = "ERROR"
    elif issue_pct > threshold_warning:
        status = "WARNING"
    else:
        status = "PASS"
    dq_log.append(Row(
        table_name="products",
        check_name=check_name,
        status=status,
        percentage=py_round(float(issue_pct), 2),
        record_count=int(issue_count)
    ))

# 1. Critical Null Check (product_id)
null_product_id = df7.filter(col("product_id").isNull()).count()
null_product_id_pct = (null_product_id / total_count) * 100
log_check("null_product_id", null_product_id, null_product_id_pct, 0, 0)

# 2. Category Missing
missing_category = df7.filter(col("is_product_category_missing") == 1).count()
missing_category_pct = (missing_category / total_count) * 100
log_check("missing_category", missing_category, missing_category_pct, None, 1)

# 3. Category Translation Check
translation_missing = df7.filter(col("is_category_translation_available") == 0).count()
translation_pct = (translation_missing / total_count) * 100
log_check("missing_category_translation", translation_missing, translation_pct, None, 2)

# 4. Product Name Length Validation
invalid_name = df7.filter(col("is_product_name_length_valid") == 0).count()
invalid_name_pct = (invalid_name / total_count) * 100
log_check("invalid_product_name_length", invalid_name, invalid_name_pct, None, 2)

# 5. Product Weight Validation
invalid_weight = df7.filter(col("is_product_weight_valid") == 0).count()
invalid_weight_pct = (invalid_weight / total_count) * 100
log_check("invalid_product_weight", invalid_weight, invalid_weight_pct, None, 2)

# 6. Product Photos Validation
invalid_photos = df7.filter(col("is_product_photos_valid") == 0).count()
invalid_photos_pct = (invalid_photos / total_count) * 100
log_check("invalid_product_photos", invalid_photos, invalid_photos_pct, None, 2)

# 7. Product Dimension Validation
invalid_dimensions = df7.filter(col("is_product_dimension_valid") == 0).count()
invalid_dim_pct = (invalid_dimensions / total_count) * 100
log_check("invalid_product_dimensions", invalid_dimensions, invalid_dim_pct, None, 2)

# Write to audit table
dq_df = spark.createDataFrame(dq_log) \
    .withColumn("run_id", lit(run_id)) \
    .withColumn("run_timestamp", current_timestamp()) \
    .withColumn("run_type", lit(run_type)) \
    .withColumn("run_date", to_date(current_timestamp()))

dq_df.write.mode("append").saveAsTable("Olist_Gold_Lakehouse.gold_data_quality_log")

# Print warnings
for r in dq_log:
    if r.status == "WARNING":
        print(f"⚠️  {r.check_name}: {r.percentage:.2f}% ({r.record_count} records)")

# Final validation
error_count = dq_df.filter(col("status") == "ERROR").count()
if error_count > 0:
    error_checks = [r.check_name for r in dq_log if r.status == "ERROR"]
    raise Exception(f"Data Quality Failed for products table: {' | '.join(error_checks)}")
else:
    print("✅ Products Data Quality Passed")

StatementMeta(, ad7a205a-e72c-4e03-b92e-52056cd52127, 11, Finished, Available, Finished, False)

RUN ID: 2026-05-04 05:38:16
⚠️  missing_category: 1.85% (610 records)
✅ Products Data Quality Passed


#### **Table : olist_sellers**

In [10]:
from pyspark.sql import Row
from pyspark.sql.functions import col, count as spark_count, current_timestamp
from datetime import datetime
from builtins import round as py_round
print("RUN ID:", run_id)
dq_log = []

total_count = df8.count()
if total_count == 0:
    raise Exception("Seller table is empty")

def log_check(check_name, issue_count, issue_pct, threshold_error, threshold_warning):
    if threshold_error is not None and issue_pct > threshold_error:
        status = "ERROR"
    elif issue_pct > threshold_warning:
        status = "WARNING"
    else:
        status = "PASS"
    dq_log.append(Row(
        table_name="sellers",
        check_name=check_name,
        status=status,
        percentage=py_round(float(issue_pct), 2),
        record_count=int(issue_count)
    ))

# 1. Critical Null Check
null_id = df8.filter(col("seller_id").isNull()).count()
null_id_pct = (null_id / total_count) * 100
log_check("null_seller_id", null_id, null_id_pct, 0, 0)

# 2. Attribute Null Checks
null_attr = df8.filter(
    col("seller_zip_code_prefix").isNull() |
    col("seller_city").isNull() |
    col("seller_state").isNull()
).count()
null_attr_pct = (null_attr / total_count) * 100
log_check("null_seller_attributes", null_attr, null_attr_pct, None, 0)

# 3. Duplicate Check
duplicate_count = df8.groupBy("seller_id") \
    .agg(spark_count("*").alias("cnt")) \
    .filter(col("cnt") > 1).count()
duplicate_pct = (duplicate_count / total_count) * 100
log_check("duplicate_seller_id", duplicate_count, duplicate_pct, 1, 0)

# 4. Location Validation
invalid_location = df8.filter(
    (col("is_state_valid") == 0) |
    (col("is_city_valid") == 0) |
    (col("is_state_city_valid") == 0)
).count()
invalid_location_pct = (invalid_location / total_count) * 100
log_check("invalid_seller_location", invalid_location, invalid_location_pct, None, 0)

# Write to audit table
dq_df = spark.createDataFrame(dq_log) \
    .withColumn("run_id", lit(run_id)) \
    .withColumn("run_timestamp", current_timestamp()) \
    .withColumn("run_type", lit(run_type)) \
    .withColumn("run_date", to_date(current_timestamp()))

dq_df.write.mode("append").saveAsTable("Olist_Gold_Lakehouse.gold_data_quality_log")

# Print warnings
for r in dq_log:
    if r.status == "WARNING":
        print(f"⚠️  {r.check_name}: {r.percentage:.2f}% ({r.record_count} records)")

# Final validation
error_count = dq_df.filter(col("status") == "ERROR").count()
if error_count > 0:
    error_checks = [r.check_name for r in dq_log if r.status == "ERROR"]
    raise Exception(f"Data Quality Failed for sellers table: {' | '.join(error_checks)}")
else:
    print("✅ Sellers Data Quality Passed")

StatementMeta(, ad7a205a-e72c-4e03-b92e-52056cd52127, 12, Finished, Available, Finished, False)

RUN ID: 2026-05-04 05:38:16
⚠️  invalid_seller_location: 2.88% (89 records)
✅ Sellers Data Quality Passed


#### **Table : olist_geolocation**

In [11]:
from pyspark.sql import Row
from pyspark.sql.functions import col, count as spark_count, current_timestamp
from datetime import datetime
from builtins import round as py_round
print("RUN ID:", run_id)
dq_log = []

# -------------------------------
# Total count
# -------------------------------
total_count = df2.count()

if total_count == 0:
    raise Exception("Geolocation table is empty")

# -------------------------------
# Helper function
# -------------------------------
def log_check(check_name, issue_count, issue_pct, threshold_error, threshold_warning):
    if threshold_error is not None and issue_pct > threshold_error:
        status = "ERROR"
    elif issue_pct > threshold_warning:
        status = "WARNING"
    else:
        status = "PASS"
    dq_log.append(Row(
        table_name="geolocation",
        check_name=check_name,
        status=status,
        percentage=py_round(float(issue_pct), 2),
        record_count=int(issue_count)
    ))

# -------------------------------
# 1. Critical Null Check (zip)
# -------------------------------
null_zip = df2.filter(col("geolocation_zip_code_prefix").isNull()).count()
null_zip_pct = (null_zip / total_count) * 100
log_check("null_zip_code", null_zip, null_zip_pct, 0, 0)

# -------------------------------
# 2. Lat/Lng Null Check
# -------------------------------
null_geo = df2.filter(
    col("geolocation_lat").isNull() |
    col("geolocation_lng").isNull()
).count()
null_geo_pct = (null_geo / total_count) * 100
log_check("missing_coordinates", null_geo, null_geo_pct, 5, 2)

# -------------------------------
# 3. City/State Null Check
# -------------------------------
null_location = df2.filter(
    col("geolocation_city").isNull() |
    col("geolocation_state").isNull()
).count()
null_location_pct = (null_location / total_count) * 100
log_check("null_city_state", null_location, null_location_pct, None, 0)

# -------------------------------
# 4. Duplicate Check
# -------------------------------
zip_state_conflict = df2.groupBy("geolocation_zip_code_prefix") \
    .agg(countDistinct("geolocation_state").alias("state_count")) \
    .filter(col("state_count") > 1) \
    .count()

total_zips = df2.select("geolocation_zip_code_prefix").distinct().count()
zip_state_pct = (zip_state_conflict / total_zips) * 100
log_check("zip_mapped_to_multiple_states", zip_state_conflict, zip_state_pct, 0.5, 0)

# -------------------------------
# 5. Location Validation (flags)
# -------------------------------
invalid_location = df2.filter(
    (col("is_state_valid") == 0) |
    (col("is_city_valid") == 0) |
    (col("is_state_city_valid") == 0)
).count()
invalid_location_pct = (invalid_location / total_count) * 100
log_check("invalid_geo_mapping", invalid_location, invalid_location_pct, None, 1)

# -------------------------------
# Write to audit table
# -------------------------------
dq_df = spark.createDataFrame(dq_log) \
    .withColumn("run_id", lit(run_id)) \
    .withColumn("run_timestamp", current_timestamp()) \
    .withColumn("run_type", lit(run_type)) \
    .withColumn("run_date", to_date(current_timestamp()))

dq_df.write.mode("append").saveAsTable("Olist_Gold_Lakehouse.gold_data_quality_log")

# -------------------------------
# Print warnings
# -------------------------------
for r in dq_log:
    if r.status == "WARNING":
        print(f"⚠️  {r.check_name}: {r.percentage:.2f}% ({r.record_count} records)")

# -------------------------------
# Final validation
# -------------------------------
error_count = dq_df.filter(col("status") == "ERROR").count()
if error_count > 0:
    error_checks = [r.check_name for r in dq_log if r.status == "ERROR"]
    raise Exception(f"Data Quality Failed for geolocation table: {' | '.join(error_checks)}")
else:
    print("✅ Geolocation Data Quality Passed")

StatementMeta(, ad7a205a-e72c-4e03-b92e-52056cd52127, 13, Finished, Available, Finished, False)

RUN ID: 2026-05-04 05:38:16
⚠️  zip_mapped_to_multiple_states: 0.04% (8 records)
✅ Geolocation Data Quality Passed


#### **Table : Product_category_translation**

In [12]:
import builtins
from pyspark.sql import Row
from pyspark.sql.functions import col, trim, lower, count as spark_count, current_timestamp
from datetime import datetime
from builtins import round as py_round
print("RUN ID:", run_id)
dq_log = []

# -------------------------------
# Total count
# -------------------------------
total_count = df9.count()

if total_count == 0:
    raise Exception("Category translation table is empty")

# -------------------------------
# Helper function
# -------------------------------
def log_check(check_name, issue_count, issue_pct, threshold_error, threshold_warning):
    if threshold_error is not None and issue_pct > threshold_error:
        status = "ERROR"
    elif issue_pct > threshold_warning:
        status = "WARNING"
    else:
        status = "PASS"
    dq_log.append(Row(
        table_name="category_translation",
        check_name=check_name,
        status=status,
        percentage=py_round(float(issue_pct), 2),
        record_count=int(issue_count)
    ))

# -------------------------------
# 1. Null Checks
# -------------------------------
null_count = df9.filter(
    col("product_category_name").isNull() |
    col("product_category_name_english").isNull()
).count()
null_pct = (null_count / total_count) * 100
log_check("null_category_mapping", null_count, null_pct, 1, 0)

# -------------------------------
# 2. Duplicate Check
# Should be 1:1 mapping
# -------------------------------
dup_pt = df9.groupBy("product_category_name") \
    .agg(spark_count("*").alias("cnt")) \
    .filter(col("cnt") > 1).count()

dup_en = df9.groupBy("product_category_name_english") \
    .agg(spark_count("*").alias("cnt")) \
    .filter(col("cnt") > 1).count()

dup_total = dup_pt + dup_en
dup_pct = (dup_total / total_count) * 100
log_check("duplicate_category_mapping", dup_total, dup_pct, 0, 0)

# -------------------------------
# 3. Referential Integrity
# Product → Category
# -------------------------------
product_clean = df7.withColumn(
    "cat_clean", trim(lower(col("product_category_name")))
)

category_clean = df9.withColumn(
    "cat_clean", trim(lower(col("product_category_name")))
)

product_total = product_clean.count()

missing_mapping = product_clean.join(
    category_clean, "cat_clean", "left_anti"
).filter(col("cat_clean").isNotNull()).count()

missing_pct = (missing_mapping / product_total) * 100
log_check("missing_category_mapping_in_products", missing_mapping, missing_pct, 5, 0)

# -------------------------------
# Write to audit table
# -------------------------------
dq_df = spark.createDataFrame(dq_log) \
    .withColumn("run_id", lit(run_id)) \
    .withColumn("run_timestamp", current_timestamp()) \
    .withColumn("run_type", lit(run_type)) \
    .withColumn("run_date", to_date(current_timestamp()))

dq_df.write.mode("append").saveAsTable("Olist_Gold_Lakehouse.gold_data_quality_log")

# -------------------------------
# Print warnings
# -------------------------------
for r in dq_log:
    if r.status == "WARNING":
        print(f"⚠️  {r.check_name}: {r.percentage:.2f}% ({r.record_count} records)")

# -------------------------------
# Final validation
# -------------------------------
error_count = dq_df.filter(col("status") == "ERROR").count()
if error_count > 0:
    error_checks = [r.check_name for r in dq_log if r.status == "ERROR"]
    raise Exception(f"Data Quality Failed for category_translation table: {' | '.join(error_checks)}")
else:
    print("✅ Category Translation Data Quality Passed")

StatementMeta(, ad7a205a-e72c-4e03-b92e-52056cd52127, 14, Finished, Available, Finished, False)

RUN ID: 2026-05-04 05:38:16
⚠️  missing_category_mapping_in_products: 0.04% (13 records)
✅ Category Translation Data Quality Passed


In [13]:
# from pyspark.sql import Row
# from pyspark.sql.functions import col, current_timestamp
# from datetime import datetime
# from builtins import round as py_round

# dq_log = []

# def log_check(check_name, issue_count, issue_pct, threshold_error, threshold_warning):
#     if threshold_error is not None and issue_pct > threshold_error:
#         status = "ERROR"
#     elif issue_pct > threshold_warning:
#         status = "WARNING"
#     else:
#         status = "PASS"
#     dq_log.append(Row(
#         table_name="customers",
#         check_name=check_name,
#         status=status,
#         percentage=py_round(float(issue_pct), 2),
#         record_count=int(issue_count),
#         run_timestamp=datetime.now()
#     ))

# log_check("null_customer_id",            null_id,          null_id_pct,          1,    0)
# log_check("null_customer_attributes",    null_attr,        null_attr_pct,        None, 0)
# log_check("duplicate_customer_id",       dup_id,           dup_id_pct,           1,    0)
# log_check("duplicate_customer_unique_id",dup_unique,       dup_unique_pct,       None, 0)
# log_check("invalid_customer_location",   invalid_location, invalid_location_pct, None, 0)

# # Write to audit table
# dq_df = spark.createDataFrame(dq_log) \
#     .withColumn("run_timestamp", current_timestamp())
# dq_df.write.mode("append").saveAsTable("Olist_Gold_Lakehouse.gold_data_quality_log")

StatementMeta(, ad7a205a-e72c-4e03-b92e-52056cd52127, 15, Finished, Available, Finished, False)

In [14]:
# from pyspark.sql import Row
# from pyspark.sql.functions import col, sum as spark_sum, count as spark_count, current_timestamp
# from datetime import datetime
# from builtins import round as py_round

# dq_log = []

# def log_check(check_name, issue_count, issue_pct, threshold_error, threshold_warning):
#     if threshold_error is not None and issue_pct > threshold_error:
#         status = "ERROR"
#     elif issue_pct > threshold_warning:
#         status = "WARNING"
#     else:
#         status = "PASS"
#     dq_log.append(Row(
#         table_name="order_items",
#         check_name=check_name,
#         status=status,
#         percentage=py_round(float(issue_pct), 2),
#         record_count=int(issue_count),
#         run_timestamp=datetime.now()
#     ))

# log_check("null_check",                          null_id,               null_id_pct,         1,    0)
# log_check("invalid_price",                       invalid_price,         price_pct,            3,    0)
# log_check("invalid_freight",                     invalid_freight,       freight_pct,          3,    0)
# log_check("delivered_price_missing",             delivered_price_missing, delivered_price_pct, 1,   0)
# log_check("duplicate_check",                     dup_count,             dup_pct,              1,    0)
# log_check("missing_payment_for_completed_orders", invalid_payment,      payment_pct,          1,    0)

# # -------------------------------
# # Write to audit table
# # -------------------------------
# dq_df = spark.createDataFrame(dq_log) \
#     .withColumn("run_timestamp", current_timestamp())
# dq_df.write.mode("append").saveAsTable("Olist_Gold_Lakehouse.gold_data_quality_log")

StatementMeta(, ad7a205a-e72c-4e03-b92e-52056cd52127, 16, Finished, Available, Finished, False)

In [15]:
# from pyspark.sql import Row
# from pyspark.sql.functions import col, current_timestamp
# from datetime import datetime
# from builtins import round as py_round

# dq_log = []

# def log_check(check_name, issue_count, issue_pct, threshold_error, threshold_warning):
#     if threshold_error is not None and issue_pct > threshold_error:
#         status = "ERROR"
#     elif issue_pct > threshold_warning:
#         status = "WARNING"
#     else:
#         status = "PASS"
#     dq_log.append(Row(
#         table_name="orders",
#         check_name=check_name,
#         status=status,
#         percentage=py_round(float(issue_pct), 2),
#         record_count=int(issue_count),
#         run_timestamp=datetime.now()
#     ))

# log_check("null_order_id",                        null_order_id,    null_order_id_pct,    1, 0)
# log_check("null_customer_id",                     null_customer_id, null_customer_id_pct, 1, 0)
# log_check("duplicate_order_id",                   duplicate_count,  duplicate_pct,        1, 0)
# log_check("purchase_after_approval",              invalid_purchase, purchase_pct,         5, 0)
# log_check("approval_after_carrier",               invalid_approval, approval_pct,         5, 0)
# log_check("carrier_after_customer",               invalid_shipping, shipping_pct,         5, 0)
# log_check("missing_payment_for_completed_orders", missing_payment,  missing_payment_pct,  1, 0)

# # Write to audit table
# dq_df = spark.createDataFrame(dq_log) \
#     .withColumn("run_timestamp", current_timestamp())
# dq_df.write.mode("append").saveAsTable("Olist_Gold_Lakehouse.gold_data_quality_log")

# # Print warnings
# for r in dq_log:
#     if r.status == "WARNING":
#         print(f"⚠️  {r.check_name}: {r.percentage:.2f}% ({r.record_count} records)")

# # Final validation
# error_count = dq_df.filter(col("status") == "ERROR").count()
# if error_count > 0:
#     error_checks = [r.check_name for r in dq_log if r.status == "ERROR"]
#     raise Exception(f"Data Quality Failed for orders table: {' | '.join(error_checks)}")
# else:
#     print("✅ Orders Data Quality Passed")

StatementMeta(, ad7a205a-e72c-4e03-b92e-52056cd52127, 17, Finished, Available, Finished, False)

In [16]:
# from pyspark.sql import Row
# from pyspark.sql.functions import col, current_timestamp
# from datetime import datetime
# from builtins import round as py_round

# dq_log = []

# def log_check(check_name, issue_count, issue_pct, threshold_error, threshold_warning):
#     if threshold_error is not None and issue_pct > threshold_error:
#         status = "ERROR"
#     elif issue_pct > threshold_warning:
#         status = "WARNING"
#     else:
#         status = "PASS"
#     dq_log.append(Row(
#         table_name="payments",
#         check_name=check_name,
#         status=status,
#         percentage=py_round(float(issue_pct), 2),
#         record_count=int(issue_count),
#         run_timestamp=datetime.now()
#     ))

# log_check("null_check",                    null_count,  null_pct,      1,    0)
# log_check("duplicate_check",               duplicate_count, duplicate_pct, 1, 0)
# log_check("payment_sequence_consistency",  seq_issue,   seq_pct,       None, 0)
# log_check("missing_orders",                ref_issue,   ref_pct,       1,    0)
# log_check("negative_payment_values",       neg_count,   neg_pct,       0,    -1)
# log_check("revenue_validation",            diff_count,  diff_pct,      None, 0)

# # Write to audit table
# dq_df = spark.createDataFrame(dq_log) \
#     .withColumn("run_timestamp", current_timestamp())
# dq_df.write.mode("append").saveAsTable("Olist_Gold_Lakehouse.gold_data_quality_log")

StatementMeta(, ad7a205a-e72c-4e03-b92e-52056cd52127, 18, Finished, Available, Finished, False)

In [17]:
# from pyspark.sql import Row
# from pyspark.sql.functions import col, current_timestamp
# from datetime import datetime
# from builtins import round as py_round

# dq_log = []

# def log_check(check_name, issue_count, issue_pct, threshold_error, threshold_warning):
#     if threshold_error is not None and issue_pct > threshold_error:
#         status = "ERROR"
#     elif issue_pct > threshold_warning:
#         status = "WARNING"
#     else:
#         status = "PASS"
#     dq_log.append(Row(
#         table_name="reviews",
#         check_name=check_name,
#         status=status,
#         percentage=py_round(float(issue_pct), 2),
#         record_count=int(issue_count),
#         run_timestamp=datetime.now()
#     ))

# log_check("null_critical_fields",          null_critical,     null_critical_pct, 1,    0)
# log_check("null_review_dates",             null_dates,        null_dates_pct,    None, 0)
# log_check("invalid_review_date_sequence",  invalid_date_flag, invalid_date_pct,  1,    0)
# log_check("invalid_review_score",          invalid_score_flag,invalid_score_pct, 1,    0)

# dq_df = spark.createDataFrame(dq_log) \
#     .withColumn("run_timestamp", current_timestamp())
# dq_df.write.mode("append").saveAsTable("Olist_Gold_Lakehouse.gold_data_quality_log")

StatementMeta(, ad7a205a-e72c-4e03-b92e-52056cd52127, 19, Finished, Available, Finished, False)

In [18]:
# from pyspark.sql import Row
# from pyspark.sql.functions import col, current_timestamp
# from datetime import datetime
# from builtins import round as py_round

# dq_log = []

# def log_check(check_name, issue_count, issue_pct, threshold_error, threshold_warning):
#     if threshold_error is not None and issue_pct > threshold_error:
#         status = "ERROR"
#     elif issue_pct > threshold_warning:
#         status = "WARNING"
#     else:
#         status = "PASS"
#     dq_log.append(Row(
#         table_name="products",
#         check_name=check_name,
#         status=status,
#         percentage=py_round(float(issue_pct), 2),
#         record_count=int(issue_count),
#         run_timestamp=datetime.now()
#     ))

# log_check("null_product_id",              null_product_id,   null_product_id_pct,  0,    -1)
# log_check("missing_category",             missing_category,  missing_category_pct, None, 0)
# log_check("missing_category_translation", translation_missing, translation_pct,   None, 0)
# log_check("invalid_product_name_length",  invalid_name,      invalid_name_pct,     None, 0)
# log_check("invalid_product_weight",       invalid_weight,    invalid_weight_pct,   None, 0)
# log_check("invalid_product_photos",       invalid_photos,    invalid_photos_pct,   None, 0)
# log_check("invalid_product_dimensions",   invalid_dimensions,invalid_dim_pct,      None, 0)

# dq_df = spark.createDataFrame(dq_log) \
#     .withColumn("run_timestamp", current_timestamp())
# dq_df.write.mode("append").saveAsTable("Olist_Gold_Lakehouse.gold_data_quality_log")

StatementMeta(, ad7a205a-e72c-4e03-b92e-52056cd52127, 20, Finished, Available, Finished, False)

In [19]:
# from pyspark.sql import Row
# from pyspark.sql.functions import col, current_timestamp
# from datetime import datetime
# from builtins import round as py_round

# dq_log = []

# def log_check(check_name, issue_count, issue_pct, threshold_error, threshold_warning):
#     if threshold_error is not None and issue_pct > threshold_error:
#         status = "ERROR"
#     elif issue_pct > threshold_warning:
#         status = "WARNING"
#     else:
#         status = "PASS"
#     dq_log.append(Row(
#         table_name="sellers",
#         check_name=check_name,
#         status=status,
#         percentage=py_round(float(issue_pct), 2),
#         record_count=int(issue_count),
#         run_timestamp=datetime.now()
#     ))

# log_check("null_seller_id",          null_id,          null_id_pct,          1,    0)
# log_check("null_seller_attributes",  null_attr,        null_attr_pct,        None, 0)
# log_check("duplicate_seller_id",     duplicate_count,  duplicate_pct,        1,    0)
# log_check("invalid_seller_location", invalid_location, invalid_location_pct, None, 0)

# dq_df = spark.createDataFrame(dq_log) \
#     .withColumn("run_timestamp", current_timestamp())
# dq_df.write.mode("append").saveAsTable("Olist_Gold_Lakehouse.gold_data_quality_log")

StatementMeta(, ad7a205a-e72c-4e03-b92e-52056cd52127, 21, Finished, Available, Finished, False)

In [20]:
# from pyspark.sql import Row
# from pyspark.sql.functions import col, current_timestamp
# from datetime import datetime
# from builtins import round as py_round

# dq_log = []

# def log_check(check_name, issue_count, issue_pct, threshold_error, threshold_warning):
#     if threshold_error is not None and issue_pct > threshold_error:
#         status = "ERROR"
#     elif issue_pct > threshold_warning:
#         status = "WARNING"
#     else:
#         status = "PASS"
#     dq_log.append(Row(
#         table_name="geolocation",
#         check_name=check_name,
#         status=status,
#         percentage=py_round(float(issue_pct), 2),
#         record_count=int(issue_count),
#         run_timestamp=datetime.now()
#     ))

# log_check("null_zip_code",                  null_zip,          null_zip_pct,          1,    0)
# log_check("missing_coordinates",            null_geo,          null_geo_pct,          1,    0)
# log_check("null_city_state",                null_location,     null_location_pct,     None, 0)
# log_check("zip_mapped_to_multiple_states",  zip_state_conflict,zip_state_pct,         1,    0)
# log_check("invalid_geo_mapping",            invalid_location,  invalid_location_pct,  None, 0)

# dq_df = spark.createDataFrame(dq_log) \
#     .withColumn("run_timestamp", current_timestamp())
# dq_df.write.mode("append").saveAsTable("Olist_Gold_Lakehouse.gold_data_quality_log")

StatementMeta(, ad7a205a-e72c-4e03-b92e-52056cd52127, 22, Finished, Available, Finished, False)

In [21]:
# from pyspark.sql import Row
# from pyspark.sql.functions import col, current_timestamp
# from datetime import datetime
# from builtins import round as py_round

# dq_log = []

# def log_check(check_name, issue_count, issue_pct, threshold_error, threshold_warning):
#     if threshold_error is not None and issue_pct > threshold_error:
#         status = "ERROR"
#     elif issue_pct > threshold_warning:
#         status = "WARNING"
#     else:
#         status = "PASS"
#     dq_log.append(Row(
#         table_name="category_translation",
#         check_name=check_name,
#         status=status,
#         percentage=py_round(float(issue_pct), 2),
#         record_count=int(issue_count),
#         run_timestamp=datetime.now()
#     ))

# log_check("null_category_mapping",              null_count,      null_pct,     1,    0)
# log_check("duplicate_category_mapping",         dup_total,       dup_pct,      0,    -1)
# log_check("missing_category_mapping_in_products",missing_mapping, missing_pct, 5,    0)

# dq_df = spark.createDataFrame(dq_log) \
#     .withColumn("run_timestamp", current_timestamp())
# dq_df.write.mode("append").saveAsTable("Olist_Gold_Lakehouse.gold_data_quality_log")

StatementMeta(, ad7a205a-e72c-4e03-b92e-52056cd52127, 23, Finished, Available, Finished, False)